# Supervised Learning for Operations Research

Predict-then-Optimize pipelines using regression and classification models to support decision-making under uncertainty.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report

sns.set(style="whitegrid")
np.random.seed(42)

## Demand Forecasting

In [ ]:
# Generate realistic synthetic demand dataset for inventory / production planning
n_samples = 800

data = pd.DataFrame({
    'week': np.arange(1, n_samples + 1),
    'price': np.random.uniform(15, 60, n_samples),
    'promotion': np.random.binomial(1, 0.35, n_samples),
    'holiday': np.random.binomial(1, 0.12, n_samples),
    'competitor_price': np.random.uniform(18, 65, n_samples),
    'temperature': np.random.normal(18, 10, n_samples),
    'marketing_spend': np.random.uniform(500, 5000, n_samples)
})

# Realistic demand function with noise
data['demand'] = (
    180 
    - 2.8 * data['price'] 
    + 65 * data['promotion'] 
    + 95 * data['holiday'] 
    - 1.8 * (data['price'] - data['competitor_price'])
    + 2.2 * data['temperature']
    + 0.012 * data['marketing_spend']
    + np.random.normal(0, 30, n_samples)
)

data['demand'] = data['demand'].clip(lower=10)

print("Dataset preview:")
display(data.head())

In [ ]:
# Train-test split and model
X = data.drop(['demand', 'week'], axis=1)
y = data['demand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf_reg = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)

y_pred = rf_reg.predict(X_test)

print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"Root Mean Squared Error: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

In [ ]:
# Feature importance
importances = pd.Series(rf_reg.feature_importances_, index=X.columns).sort_values(ascending=True)
importances.plot(kind='barh', figsize=(8, 6), color='steelblue')
plt.title('Feature Importance - Demand Prediction')
plt.xlabel('Importance')
plt.show()

## Predict-then-Optimize

In [ ]:
# Newsvendor parameters
c_o = 3.0   # overage (holding) cost
c_u = 12.0  # underage (lost sales) cost

# Predict demand for a future scenario
future_scenario = pd.DataFrame({
    'price': [42],
    'promotion': [1],
    'holiday': [0],
    'competitor_price': [45],
    'temperature': [25],
    'marketing_spend': [3200]
})

predicted_demand = rf_reg.predict(future_scenario)[0]
print(f"Predicted weekly demand: {predicted_demand:.1f}")

# Critical fractile solution (deterministic approximation)
critical_ratio = c_u / (c_o + c_u)
recommended_order = predicted_demand  # can be extended with safety stock

print(f"Recommended order quantity: {recommended_order:.0f} units")

## Classification for Revenue Management

In [ ]:
# Synthetic booking / offer acceptance dataset
n = 1200
booking_data = pd.DataFrame({
    'lead_time_days': np.random.randint(1, 90, n),
    'fare_class': np.random.randint(1, 6, n),
    'day_of_week': np.random.randint(0, 7, n),
    'remaining_capacity': np.random.randint(5, 250, n),
    'price_sensitivity': np.random.uniform(0.4, 1.8, n)
})

# Target: accept offer or not
booking_data['accept'] = (
    (booking_data['lead_time_days'] < 35) &
    (booking_data['fare_class'] >= 3) &
    (booking_data['remaining_capacity'] > 40) &
    (booking_data['price_sensitivity'] < 1.2)
).astype(int)

# Add some noise
booking_data['accept'] = booking_data['accept'] ^ np.random.binomial(1, 0.18, n)

X_cls = booking_data.drop('accept', axis=1)
y_cls = booking_data['accept']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.25, random_state=42)

rf_clf = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_c, y_train_c)

y_pred_c = rf_clf.predict(X_test_c)

print(classification_report(y_test_c, y_pred_c))

## Exercises

- Replace `RandomForestRegressor` with `XGBoost` or a neural network using PyTorch.
- Integrate the predicted demand distribution into a full stochastic program.
- Add cross-validation and hyperparameter tuning with `GridSearchCV` or `Optuna`.

The next chapter will cover **Unsupervised Learning** techniques such as clustering for customer segmentation and facility location problems.